In [2]:
from pathlib import Path
import geopandas as gpd
import os
import osmium
from osmium.filter import EmptyTagFilter, TagFilter, GeoInterfaceFilter

In [3]:
geofabrik_path = Path(os.environ["GEOFABRIK_PATH"])

In [4]:
data = (
    osmium.FileProcessor(geofabrik_path / "mexico-260407.osm.pbf")
    .with_areas()
    .with_filter(EmptyTagFilter())
    .with_filter(
        TagFilter(
            ("leisure", "park")
        )
    )
    .with_filter(GeoInterfaceFilter(drop_invalid_geometries=True))
)

df = gpd.GeoDataFrame.from_features(data, crs="EPSG:4326").drop(columns=["colonia"])

In [7]:
df.loc[lambda x: x["geometry"].geom_type.isin(["Polygon", "MultiPolygon"])]

,geometry,addr:city,addr:housenumber,addr:postcode,addr:street,leisure,name,opening_hours,population,website,...,supervised,pets_allowed,operational_status,status,bench,bridge,operator:wikipedia,description:es,contact:instagram,fence
1725,"MULTIPOLYGON (((-99.17072 19.41093, -99.1707 1...",Ciudad de México,NaN,06100,Avenida México,park,Parque México,24/7,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1726,"MULTIPOLYGON (((-99.1462 19.43519, -99.14489 1...",NaN,NaN,NaN,NaN,park,Alameda Central,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1727,"MULTIPOLYGON (((-99.16962 19.3706, -99.16952 1...",NaN,NaN,NaN,NaN,park,Parque Pascual Ortiz Rubio,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1728,"MULTIPOLYGON (((-99.17764 19.37558, -99.17753 ...",NaN,NaN,NaN,NaN,park,Parque San Lorenzo,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1729,"MULTIPOLYGON (((-99.17269 19.37831, -99.17247 ...",NaN,NaN,NaN,NaN,park,Jardín del Arte Tlacoquemécatl,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
89899,"MULTIPOLYGON (((-106.12616 28.63896, -106.1261...",NaN,NaN,NaN,NaN,park,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
89900,"MULTIPOLYGON (((-106.1269 28.63958, -106.12679...",NaN,NaN,NaN,NaN,park,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
89901,"MULTIPOLYGON (((-106.13056 28.64046, -106.1304...",NaN,NaN,NaN,NaN,park,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
89902,"MULTIPOLYGON (((-106.13139 28.64016, -106.1313...",NaN,NaN,NaN,NaN,park,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [22]:
mask_poly = df["geometry"].geom_type.isin(["Polygon", "MultiPolygon"])
mask_points_with_area = df["geometry"].geom_type.isin(["Point"]) & df["area"].notna() & df["area"] > 0
temp = df.loc[mask_poly | mask_points_with_area]
temp[temp["geometry"].apply(lambda x: x.geoms).str.len() > 1].to_file("multipoly.gpkg")

In [17]:
temp.explode()

,addr:city,addr:housenumber,addr:postcode,addr:street,leisure,name,opening_hours,population,website,Colonia,...,pets_allowed,operational_status,status,bench,bridge,operator:wikipedia,description:es,contact:instagram,fence,geometry
1725,Ciudad de México,NaN,06100,Avenida México,park,Parque México,24/7,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((-99.17072 19.41093, -99.1707 19.4108..."
1726,NaN,NaN,NaN,NaN,park,Alameda Central,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((-99.1462 19.43519, -99.14489 19.4349..."
1727,NaN,NaN,NaN,NaN,park,Parque Pascual Ortiz Rubio,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((-99.16962 19.3706, -99.16952 19.3705..."
1728,NaN,NaN,NaN,NaN,park,Parque San Lorenzo,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((-99.17764 19.37558, -99.17753 19.375..."
1729,NaN,NaN,NaN,NaN,park,Jardín del Arte Tlacoquemécatl,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((-99.17269 19.37831, -99.17247 19.378..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
89899,NaN,NaN,NaN,NaN,park,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((-106.12616 28.63896, -106.12611 28.6..."
89900,NaN,NaN,NaN,NaN,park,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((-106.1269 28.63958, -106.12679 28.63..."
89901,NaN,NaN,NaN,NaN,park,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((-106.13056 28.64046, -106.13044 28.6..."
89902,NaN,NaN,NaN,NaN,park,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"POLYGON ((-106.13139 28.64016, -106.13132 28.6..."
